# 🎬 Netflix Data Analysis

**Tools:** Python • Pandas • NumPy • Matplotlib • Seaborn

## Objective
Analyze Netflix Movies and TV Shows to discover patterns in content type, release years, ratings, genres, countries, duration, and content-added trends.


## Business Questions
1. Movies vs TV Shows — which dominates the catalog?
2. How has content changed over time?
3. Which countries produce the most titles?
4. What genres are most common?
5. Which ratings are most frequent?
6. What is the typical movie duration?
7. In which years was the most content added?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)


In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/japnitahuja/netflix-data-analysis/main/netflix_titles.csv'
LOCAL_PATH = Path('data/netflix_titles.csv')
df = pd.read_csv(LOCAL_PATH) if LOCAL_PATH.exists() else pd.read_csv(DATA_URL)
df.head()

## 1. Data Understanding

In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.describe(include='all').T)

## 2. Missing-Value Analysis

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_report[missing_report['missing_count'] > 0]

## 3. Data Cleaning & Feature Engineering

In [ ]:
df = df.drop_duplicates().copy()
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month_name()

for col in ['director', 'cast', 'country', 'rating']:
    df[col] = df[col].fillna('Unknown')

df['duration_value'] = pd.to_numeric(
    df['duration'].astype(str).str.extract(r'(\d+)')[0], errors='coerce'
)
df['duration_unit'] = np.where(
    df['type'].eq('Movie'), 'minutes',
    np.where(df['type'].eq('TV Show'), 'seasons', 'unknown')
)

df[['title','type','date_added','year_added','duration_value','duration_unit']].head()

## 4. KPI Summary

In [ ]:
kpis = {
    'Total Titles': len(df),
    'Movies': (df['type'] == 'Movie').sum(),
    'TV Shows': (df['type'] == 'TV Show').sum(),
    'Unique Release Years': df['release_year'].nunique(),
    'Countries Represented': df['country'].nunique()
}
pd.Series(kpis)

## 5. Movies vs TV Shows

In [ ]:
type_counts = df['type'].value_counts()
display(type_counts)
plt.figure(figsize=(8,5))
sns.barplot(x=type_counts.index, y=type_counts.values)
plt.title('Netflix Content by Type')
plt.xlabel('Content Type'); plt.ylabel('Number of Titles')
plt.tight_layout(); plt.show()

## 6. Release-Year Trend

In [ ]:
yearly = df.groupby('release_year').size().reset_index(name='titles')
yearly = yearly[yearly['release_year'] >= 2000]
plt.figure(figsize=(12,5))
sns.lineplot(data=yearly, x='release_year', y='titles')
plt.title('Netflix Titles by Release Year (2000+)')
plt.xlabel('Release Year'); plt.ylabel('Number of Titles')
plt.tight_layout(); plt.show()

## 7. Top Ratings

In [ ]:
top_ratings = df['rating'].value_counts().head(10).reset_index()
top_ratings.columns = ['rating','titles']
display(top_ratings)
plt.figure(figsize=(10,5))
sns.barplot(data=top_ratings, y='rating', x='titles')
plt.title('Top Netflix Content Ratings')
plt.tight_layout(); plt.show()

## 8. Top Countries

In [ ]:
countries = df.assign(country=df['country'].str.split(', ')).explode('country')
top_countries = countries['country'].value_counts().head(10).reset_index()
top_countries.columns = ['country','titles']
display(top_countries)
plt.figure(figsize=(10,6))
sns.barplot(data=top_countries, y='country', x='titles')
plt.title('Top Countries by Number of Netflix Titles')
plt.tight_layout(); plt.show()

## 9. Top Genres

In [ ]:
genres = df.assign(genre=df['listed_in'].str.split(', ')).explode('genre')
top_genres = genres['genre'].value_counts().head(10).reset_index()
top_genres.columns = ['genre','titles']
display(top_genres)
plt.figure(figsize=(10,6))
sns.barplot(data=top_genres, y='genre', x='titles')
plt.title('Top Netflix Genres')
plt.tight_layout(); plt.show()

## 10. Movie Duration Analysis

In [ ]:
movies = df[df['type'].eq('Movie') & df['duration_value'].notna()].copy()
print(movies['duration_value'].describe())
plt.figure(figsize=(10,5))
sns.histplot(movies['duration_value'], bins=30, kde=True)
plt.title('Movie Duration Distribution')
plt.xlabel('Duration (minutes)'); plt.ylabel('Number of Movies')
plt.tight_layout(); plt.show()

## 11. Content Added to Netflix Over Time

In [ ]:
added = df.dropna(subset=['year_added']).groupby(['year_added','type']).size().reset_index(name='titles')
plt.figure(figsize=(12,5))
sns.lineplot(data=added, x='year_added', y='titles', hue='type', marker='o')
plt.title('Netflix Titles Added by Year')
plt.xlabel('Year Added'); plt.ylabel('Number of Titles')
plt.tight_layout(); plt.show()

## 12. Key Insights

After running the notebook, write 4–6 evidence-based insights here. Example format:
- **Content mix:** Movies/TV Shows make up the majority of the catalog — quantify the split from the KPI output.
- **Release trend:** Recent decades show a strong increase in catalog titles compared with older release years.
- **Geography:** A small group of countries contributes a large share of titles.
- **Genres:** A few broad genres dominate the catalog.
- **Ratings:** Mature/general-audience ratings account for a large portion of the catalog.
- **Duration:** Most movies cluster around a typical feature-film duration, with a long tail of shorter/longer titles.


## 13. Export Cleaned Dataset
The cleaned dataset can be used for a Power BI dashboard as a second project layer.

In [ ]:
Path('output').mkdir(exist_ok=True)
df.to_csv('output/netflix_titles_cleaned.csv', index=False)
print('Saved: output/netflix_titles_cleaned.csv')

## Portfolio Extension: Power BI
Use `output/netflix_titles_cleaned.csv` to build a 1-page dashboard with:
- KPI cards: Total Titles, Movies, TV Shows, Countries
- Donut chart: Movies vs TV Shows
- Line chart: Titles by Release Year
- Bar chart: Top 10 Countries
- Bar chart: Top 10 Genres
- Bar chart: Top Ratings
- Slicers: Type, Rating, Release Year, Country
